# HPD 1 — Extensión (ejercicios evaluables avanzados)

<figure>
<a
href="https://colab.research.google.com/github/Adamychen/m10_quarto/blob/main/notebooks/evaluables/hpd1-extension.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

> **Peso en la nota:** 7.5 % extra (parte del 30 % de entregas
> prácticas)
>
> **Plazo:** 14 días tras la sesión presencial.
>
> **Requisito previo:** haber completado `hpd1-evaluables.qmd`.
>
> **Entrega:** Notebook `.ipynb` ejecutado. Cada ejercicio especifica
> qué variable debe contener el resultado para la corrección automática.

> **Cómo se corrige**
>
> Cada ejercicio pide que asignes el resultado a una variable con un
> nombre concreto. El script de corrección ejecutará tu notebook e
> inspeccionará esas variables. **Si la variable no existe o tiene un
> tipo incorrecto, el ejercicio se puntúa como 0.**
>
> Para autoevaluarte antes de entregar:
>
> ``` bash
> python scripts/corregir_hpd1.py --extension tu_notebook.ipynb
> ```

In [1]:
!pip install -q chromadb sentence-transformers fpdf2 matplotlib langchain langchain-community langchain-text-splitters pypdf faiss-cpu openai deep-translator rank_bm25

In [2]:
import os
import time
import tempfile
import shutil
import faiss
import numpy as np
from fpdf import FPDF
from openai import OpenAI
from deep_translator import GoogleTranslator
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
)
import chromadb

# ------------------------------------------------------------------ Setup compartido
BASE_DIR = tempfile.mkdtemp()
CORPUS_DIR = os.path.join(BASE_DIR, "corpus")
os.makedirs(CORPUS_DIR)

MODEL_NAME = "intfloat/e5-small-v2"
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

model = SentenceTransformer(MODEL_NAME)
client = chromadb.EphemeralClient()
DIM = model.get_sentence_embedding_dimension()

print(f"Modelo: {MODEL_NAME} ({DIM} dimensiones)")
print(f"Directorio temporal: {BASE_DIR}")

# ------------------------------------------------------------------ Corpus sintético
class PDF(FPDF):
    pass

def crear_pdf(nombre, titulo, contenido):
    pdf = PDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, titulo, new_x="LMARGIN", new_y="NEXT")
    pdf.ln(5)
    pdf.set_font("Helvetica", "", 11)
    pdf.multi_cell(0, 6, contenido)
    pdf.output(os.path.join(CORPUS_DIR, nombre))

corpus = {
    "01_transformers.pdf": (
        "Transformers and Attention Mechanisms",
        "The Transformer architecture, introduced by Vaswani et al. in 2017, "
        "revolutionized natural language processing by eliminating recurrences "
        "and relying entirely on attention mechanisms. "
        "The key component is multi-head attention, which allows the model "
        "to simultaneously attend to different parts of the input sequence "
        "from different representation subspaces. Each attention head "
        "computes its own Query, Key, and Value weights through learned "
        "linear projections, and the outputs are concatenated and linearly "
        "projected. This enables capturing both local and global relationships "
        "between tokens without depending on sequential distance. Positional "
        "encoding is added to the input embeddings to preserve word order, "
        "since attention by itself is position-invariant. There are fixed "
        "sinusoidal encodings and learned positional encodings. Transformers "
        "support three main configurations: encoder-only like BERT, decoder-only "
        "like GPT and Llama, and encoder-decoder like T5. The decoder-only "
        "version is the foundation of modern large language models such as "
        "GPT-4 and Llama 3."
    ),
    "02_embeddings.pdf": (
        "Embeddings and Semantic Vector Representation",
        "Text embeddings are dense vector representations that capture the "
        "semantic meaning of words, phrases, or entire documents in a "
        "continuous vector space of fixed dimensionality. Models such as "
        "Sentence-BERT, E5, and OpenAI text-embedding-3 project texts into "
        "a space where the distance between vectors reflects semantic "
        "similarity, not lexical overlap. The most widely used metric is "
        "cosine similarity, which measures the angle between two normalized "
        "vectors independently of their magnitude. Modern embeddings are "
        "trained with contrastive objectives: maximizing similarity between "
        "semantically equivalent texts while minimizing it between unrelated "
        "ones. Embeddings are the foundation of semantic search and RAG "
        "systems, as they allow finding relevant information even when the "
        "exact words do not match between the query and the document. "
        "Models like intfloat/e5-small-v2 offer an excellent balance between "
        "quality and efficiency."
    ),
    "03_rag.pdf": (
        "RAG: Retrieval-Augmented Generation",
        "RAG is an architecture that combines information retrieval systems "
        "with generative models to produce answers grounded in documents. "
        "The typical pipeline consists of three stages: ingestion, retrieval, "
        "and generation. During ingestion, documents are split into chunks, "
        "embeddings are generated, and they are indexed in a vector database "
        "such as Chroma or FAISS. The choice of chunk size and overlap is "
        "critical: small chunks improve retrieval precision but lose context, "
        "while large chunks preserve context but dilute relevance. During "
        "retrieval, the user query is converted into an embedding and the "
        "most similar chunks are retrieved using cosine similarity search. "
        "During generation, the retrieved chunks are added to the LLM prompt. "
        "Advanced RAG incorporates techniques such as re-ranking with "
        "cross-encoders, query expansion with HyDE, and RAG-Fusion."
    ),
}

for nombre, (titulo, contenido) in corpus.items():
    crear_pdf(nombre, titulo, contenido)

# Cargar documentos con PyPDFLoader
docs = []
for filename in sorted(os.listdir(CORPUS_DIR)):
    if not filename.endswith(".pdf"):
        continue
    loader = PyPDFLoader(os.path.join(CORPUS_DIR, filename))
    docs.extend(loader.load())

print(f"Corpus listo: {len(docs)} páginas cargadas en {CORPUS_DIR}")

Modelo: intfloat/e5-small-v2 (384 dimensiones)
Directorio temporal: /var/folders/bd/_fc1rf8x6jv7w5hxv_rlkdbr0000gn/T/tmpz9b6ylxt
Corpus listo: 3 páginas cargadas en /var/folders/bd/_fc1rf8x6jv7w5hxv_rlkdbr0000gn/T/tmpz9b6ylxt/corpus

/var/folders/bd/_fc1rf8x6jv7w5hxv_rlkdbr0000gn/T/ipykernel_91813/3694618458.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = model.get_sentence_embedding_dimension()

> **Idioma del corpus**
>
> El corpus sintético está en **inglés** porque el modelo
> `intfloat/e5-small-v2` está entrenado principalmente en inglés. El
> corpus en español (`CORPUS_ES`) se usa solo como fallback para el
> **Ejercicio E1** (pipeline multilingüe con
> `intfloat/multilingual-e5-small`). En los demás ejercicios, usa el
> corpus en inglés para obtener embeddings de calidad.

In [3]:
# Corpus en español precargado (fallback para E1 si no hay internet)
CORPUS_ES = {
    "01_transformers.pdf": (
        "Transformers y Mecanismos de Atención",
        "La arquitectura Transformer, introducida por Vaswani et al. en 2017, "
        "revolucionó el procesamiento del lenguaje natural al eliminar las "
        "recurrencias y basarse completamente en mecanismos de atención. "
        "El componente clave es la atención multi-cabeza, que permite al modelo "
        "atender simultáneamente a diferentes partes de la secuencia de entrada "
        "desde distintos subespacios de representación."
    ),
    "02_embeddings.pdf": (
        "Embeddings y Representación Vectorial Semántica",
        "Los embeddings de texto son representaciones vectoriales densas que "
        "capturan el significado semántico de palabras, frases o documentos "
        "completos en un espacio vectorial continuo de dimensionalidad fija. "
        "Modelos como Sentence-BERT, E5 y OpenAI text-embedding-3 proyectan "
        "textos en un espacio donde la distancia refleja la similitud semántica."
    ),
    "03_rag.pdf": (
        "RAG: Retrieval-Augmented Generation",
        "RAG es una arquitectura que combina sistemas de recuperación de "
        "información con modelos generativos para producir respuestas "
        "fundamentadas en documentos. El pipeline típico consta de tres etapas: "
        "ingesta, recuperación y generación."
    ),
}

# ─── Para E1: usar CORPUS_ES en lugar de traducir (descomenta):
# corpus = CORPUS_ES

# ------------------------------------------------------------------ Funciones compartidas
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

def recall_at_k(textos_recuperados, palabra_clave, k):
    """recall@1: 1.0 si palabra_clave aparece en algún texto del top-k."""
    if not palabra_clave:
        return 0.0
    top_k = textos_recuperados[:k]
    return 1.0 if any(palabra_clave.lower() in t.lower() for t in top_k) else 0.0

def indexar_en_chroma(nombre_coleccion, chunks):
    col = client.get_or_create_collection(
        name=nombre_coleccion, metadata={"hnsw:space": "cosine"}
    )
    for chunk_idx, chunk in enumerate(chunks):
        emb = model.encode(chunk.page_content).tolist()
        fuente = os.path.basename(chunk.metadata["source"])
        cid = f"{fuente.replace('.pdf', '')}_chunk_{chunk_idx}"
        col.add(ids=[cid], embeddings=[emb], documents=[chunk.page_content],
                metadatas=[{"fuente": fuente, "chunk_index": chunk_idx}])
    return col

def buscar_en_chroma(collection, query, k=3):
    q_emb = model.encode(query).tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=k)
    return res

------------------------------------------------------------------------

## Ejercicio E1 — Pipeline multilingüe (2.5 puntos)

Carga el modelo `intfloat/multilingual-e5-small`. Traduce los textos de
los 3 PDFs sintéticos al español usando `GoogleTranslator` de
`deep-translator`. Indexa los textos en español en Chroma y mide:

- `ext1_recall_cross`: `recall@1` medio con **3 consultas en inglés**
  buscando sobre los documentos en español.
- `ext1_recall_mono`: `recall@1` medio con las mismas 3 consultas en
  inglés buscando sobre los documentos originales en inglés.

In [4]:
# ─── Carga el modelo multilingüe ───
# model_multi = SentenceTransformer("intfloat/multilingual-e5-small")

# ─── Traduce los textos al español ───
# Usa GoogleTranslator(source="en", target="es").translate(texto)
# Crea nuevos objetos Document con el texto traducido.
# Pista: traduce solo los chunks, no los documentos enteros.

# ─── Indexa español e inglés en colecciones separadas ───
# col_es = indexar_en_chroma(...)
# col_en = indexar_en_chroma(...)

# ─── Evalúa recall@1 con estas 3 consultas ───
# QRELS_E1: tupla (kw_es para buscar en español, kw_en para buscar en inglés)
# QRELS_E1 = {
#     "How does multi-head attention work?":            ("atención multi-cabeza", "multi-head attention"),
#     "What are text embeddings?":                      ("representaciones vectoriales densas", "dense vector representations"),
#     "How does RAG combine retrieval and generation?": ("combina sistemas de recuperación", "information retrieval systems"),
# }

ext1_recall_cross = None  # ← float: recall@1 medio inglés → español
ext1_recall_mono = None   # ← float: recall@1 medio inglés → inglés

------------------------------------------------------------------------

## Ejercicio E2 — Comparativa de splitters (2.5 puntos)

Compara `RecursiveCharacterTextSplitter` (corta por párrafos → frases →
espacios) con `CharacterTextSplitter` (corte fijo por caracteres, sin
jerarquía semántica). Usa los mismos parámetros (`chunk_size=300`,
`chunk_overlap=50`) para ambos.

Mide sobre el corpus en inglés:

- `ext2_recall_recursive`: `recall@1` medio con el splitter recursivo.
- `ext2_recall_character`: `recall@1` medio con el splitter de
  caracteres.
- `ext2_chunks`: `{"recursive": N, "character": M}` con el número de
  chunks de cada uno.

In [5]:
# ─── Crea ambos splitters ───
# recursive = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
# character = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)

# ─── Genera chunks con cada uno ───
# chunks_rec = recursive.split_documents(docs)
# chunks_char = character.split_documents(docs)

# ─── Indexa por separado y evalúa ───
# Usa indexar_en_chroma() y buscar_en_chroma().
# Ground truth por contenido (invariante al splitter):
QRELS_E2 = {
    "How does multi-head attention work?": "multi-head attention",
    "What are text embeddings?":           "dense vector representations",
    "How does RAG work?":                  "information retrieval systems",
}

ext2_recall_recursive = None  # ← float
ext2_recall_character = None  # ← float
ext2_chunks = {}              # ← {"recursive": int, "character": int}

------------------------------------------------------------------------

## Ejercicio E3 — Chroma vs FAISS (2.5 puntos)

Compara Chroma con FAISS (`IndexFlatIP`, búsqueda exacta por producto
interno). Para que el producto interno equivalga a la similitud del
coseno, **normaliza los embeddings** (divide cada vector por su norma
L2) antes de indexar en FAISS.

Usa los chunks generados con `RecursiveCharacterTextSplitter`. Mide:

- `ext3_tiempo_chroma`: segundos en indexar + buscar 1 query en Chroma.
- `ext3_tiempo_faiss`: segundos en indexar + buscar 1 query en FAISS.
- `ext3_recall_chroma`: `recall@1` medio con Chroma (3 queries).
- `ext3_recall_faiss`: `recall@1` medio con FAISS (3 queries).

Pista para FAISS:

``` python
index = faiss.IndexFlatIP(dim)        # IP = inner product
# Normalizar embeddings: emb = emb / np.linalg.norm(emb)
index.add(embeddings_normalizados)     # (n_chunks, dim)
distancias, indices = index.search(query_emb_normalizado.reshape(1, -1), k=3)
```

In [6]:
# ─── Indexa en Chroma ───
# Mide tiempo con time.perf_counter()

# ─── Indexa en FAISS ───
# 1. Normaliza embeddings (L2 norm)
# 2. Crea faiss.IndexFlatIP(dim), llama a index.add()
# 3. Normaliza la query antes de index.search()
# Mide tiempo con time.perf_counter()

# ─── Compara recall@1 (ground truth por contenido) ───
QRELS_E3 = {
    "How does multi-head attention work?": "multi-head attention",
    "What are text embeddings?":           "dense vector representations",
    "How does RAG work?":                  "information retrieval systems",
}

ext3_tiempo_chroma = None   # ← float (segundos)
ext3_tiempo_faiss = None    # ← float (segundos)
ext3_recall_chroma = None   # ← float
ext3_recall_faiss = None    # ← float

------------------------------------------------------------------------

## Ejercicio E4 — Mini-RAG con LLM (2.5 puntos)

Construye un sistema RAG mínimo: recupera los top-3 chunks para una
consulta, inyéctalos en un prompt y pide al LLM que responda usando solo
esa información.

Usa el endpoint de la asignatura (`https://llamus.cs.us.es/api/v1`), que
es compatible con la API de OpenAI. El modelo es `gemma4:e2b-mlx`. La
API key debes configurarla en la variable de entorno `LLM_API_KEY`.

In [7]:
# ─── Configura la API key ───
# import os
# os.environ["OPENAI_API_KEY"] = os.getenv("LLM_API_KEY", "not-needed")

# ─── Recupera los top-3 chunks para esta consulta ───
QUERY_E4 = "How does multi-head attention work in Transformers?"

# 1. Indexa los documentos en inglés con el pipeline ya construido.
# 2. Recupera top-3 chunks con buscar_en_chroma().
# 3. Guarda el texto de los 3 chunks en ext4_chunks.

# ─── Construye el prompt y llama al LLM ───
# client_llm = OpenAI(
#     base_url="https://llamus.cs.us.es/api/v1",
#     api_key=os.getenv("LLM_API_KEY", "not-needed"),
# )
# prompt = (
#     "Responde ÚNICAMENTE con la información proporcionada abajo. "
#     "Si la información no contiene la respuesta, di 'No lo sé'.\n\n"
#     "Contexto:\n"
#     f"{contexto}\n\n"
#     f"Pregunta: {QUERY_E4}"
# )
# respuesta = client_llm.chat.completions.create(
#     model="gemma4:e2b-mlx",
#     messages=[{"role": "user", "content": prompt}],
# ).choices[0].message.content

# ─── Evalúa manualmente la fidelidad ───
# Lee la respuesta. Si TODO lo que dice aparece en los chunks, ext4_fiel = True.
# Si se inventa algo que no está en los chunks, ext4_fiel = False.

ext4_chunks = []     # ← list[str]: texto de los 3 chunks recuperados
ext4_respuesta = ""  # ← str: respuesta del LLM
ext4_fiel = False    # ← bool: ¿la respuesta es fiel al contexto?